# Projeto Final
## Validação dos MVPs 1, 2, 3 e 4

Pipeline atual:

Finalidade
↓
Carregamento
↓
Profiling
↓
Comparação Estrutural
↓
Diagnóstico

1. MVP 1: Leitura automatizada de CSV/XLSX.
2. MVP 2: Profiling estrutural.
3. MVP 3: Comparação entre finalidade e base.
4. MVP 4: Diagnóstico de aptidão dos dados.

In [1]:
# O autoreload garante que se você alterar os arquivos .py,
# o notebook recarrega os módulos automaticamente.

%load_ext autoreload
%autoreload 2

import json
import pandas as pd

from src.loader import load_data
from src.profiler import profile_data
from src.comparator import (
    check_required_data,
    print_analise_status
)
from src.diagnostics import (
    run_diagnostic,
    print_diagnostic
)
from src.preparer import prepare_data

print("Módulos carregados com sucesso!")

Módulos carregados com sucesso!


## Execução do Pipeline com Bases Reais

Nesta etapa, o sistema é validado utilizando bases reais oriundas de fontes distintas.

Objetivos do teste:

- Validar a leitura automática de arquivos CSV/XLSX (MVP 1);
- Gerar o profiling estrutural da base (MVP 2);
- Comparar os dados encontrados com os requisitos da finalidade analítica (MVP 3);
- Diagnosticar a aptidão da base para uso analítico (MVP 4).

Finalidade utilizada no experimento:

Faturamento por Região

In [2]:
# Lista dos arquivos reais da pasta data/exemplo/

arquivos_teste = [
    "data/exemplo/vendas_kaggle.csv",
    "data/exemplo/vendas_governo.csv",
    "data/exemplo/erp.csv"
]

resultados_finais = []

for caminho in arquivos_teste:

    print(f"\n🚀 INICIANDO TESTE COM: {caminho}\n")

    try:

        # MVP 1 -------------------------
        carga = load_data(caminho)

        df = carga["df"]

        print("METADADOS DE LEITURA")
        print(carga["metadata"])
        print()

        # MVP 2 -------------------------

        perfil = profile_data(df)

        # MVP 3 -------------------------

        mapeamento = check_required_data(
            perfil,
            "faturamento_por_regiao"
        )

        print_analise_status(mapeamento)

        print()

        # MVP 4 -------------------------

        diagnostico = run_diagnostic(
            perfil,
            mapeamento
        )

        print_diagnostic(diagnostico)

        resultados_finais.append(
            {
                "arquivo": caminho,
                "status": diagnostico["status"]
            }
        )

        # MVP 5 -------------------------

        preparacao = prepare_data(
            df,
            diagnostico
        )

        base_preparada = preparacao["df"]

        print("\nTRANSFORMAÇÕES APLICADAS\n")

        for item in preparacao["transformacoes"]:
            print(item)

    except Exception as erro:

        print(
            f"❌ Erro ao processar a base: {erro}"
        )

        resultados_finais.append(
            {
                "arquivo": caminho,
                "status": diagnostico["status"],
                "transformacoes": len(
                    preparacao["transformacoes"]
                )
            }
        )

    print("\n" + "=" * 60 + "\n")

print("\nRESUMO FINAL\n")

display(
    pd.DataFrame(resultados_finais)
)


🚀 INICIANDO TESTE COM: data/exemplo/vendas_kaggle.csv

METADADOS DE LEITURA
{'nome_arquivo': 'vendas_kaggle.csv', 'caminho': 'data\\exemplo\\vendas_kaggle.csv', 'extensao': '.csv', 'encoding': 'utf-8-sig', 'delimitador': ',', 'planilha': 'n/a', 'linhas_carregadas': 1000, 'colunas_carregadas': 14}

COMPARAÇÃO COM A FINALIDADE
Finalidade: Faturamento por Região
Identificador: faturamento_por_regiao
Descrição: Avalia se a base contém os dados mínimos necessários para calcular e analisar o faturamento por região.

DADOS NECESSÁRIOS

Conceito: data_venda
Status: Encontrado
Coluna encontrada: Sale_Date
Alias reconhecido: sale date

Conceito: regiao
Status: Encontrado
Coluna encontrada: Region
Alias reconhecido: region

Conceito: valor_venda
Status: Encontrado
Coluna encontrada: Sales_Amount
Alias reconhecido: sales amount

DADOS ADICIONAIS

- Product_ID
- Sales_Rep
- Quantity_Sold
- Product_Category
- Unit_Cost
- Unit_Price
- Customer_Type
- Discount
- Payment_Method
- Sales_Channel
- Regio

,arquivo,status
0,data/exemplo/vendas_kaggle.csv,APTA
1,data/exemplo/vendas_governo.csv,REQUER PREPARAÇÃO
2,data/exemplo/erp.csv,REQUER PREPARAÇÃO


## Inspeção Profunda (Opcional)

Visualização da estrutura JSON gerada pelo MVP 2 (Profiling).

Esta etapa permite auditar os metadados produzidos pelo sistema e validar as métricas extraídas automaticamente da base.

In [3]:
caminho_inspecao = "data/exemplo/vendas_kaggle.csv"

try:

    carga = load_data(caminho_inspecao)

    df = carga["df"]

    perfil = profile_data(df)

    print(
        f"Radiografia da base: "
        f"{caminho_inspecao}\n"
    )

    json_saida = json.dumps(
        perfil,
        indent=4,
        ensure_ascii=False
    )

    if len(json_saida) > 1000:

        print(
            json_saida[:1000]
            + "\n\n... [JSON TRUNCADO PARA FACILITAR LEITURA]"
        )

    else:
        print(json_saida)

except FileNotFoundError:

    print(
        "Para inspecionar o JSON, "
        "garanta que o arquivo acima existe."
    )

Radiografia da base: data/exemplo/vendas_kaggle.csv

{
    "geral": {
        "linhas": 1000,
        "colunas": 14,
        "duplicadas": 0,
        "base_vazia": false
    },
    "colunas": [
        {
            "nome": "Product_ID",
            "tipo": "int64",
            "categoria_tipo": "numerico",
            "total": 1000,
            "preenchidos": 1000,
            "nulos": 0,
            "vazios": 0,
            "ausentes_total": 0,
            "proporcao_nulos": 0.0,
            "proporcao_vazios": 0.0,
            "proporcao_ausentes": 0.0,
            "valores_distintos": 100,
            "coluna_vazia": false
        },
        {
            "nome": "Sale_Date",
            "tipo": "object",
            "categoria_tipo": "texto",
            "total": 1000,
            "preenchidos": 1000,
            "nulos": 0,
            "vazios": 0,
            "ausentes_total": 0,
            "proporcao_nulos": 0.0,
            "proporcao_vazios": 0.0,
            "proporcao_ause